<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 01 — MLflow: Experiment Tracking

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~2h30

</div>

## 📋 Lab Overview

### Learning Objectives

By the end of this lab, you will be able to:

1. **Launch and connect to an MLflow tracking server**
2. **Log parameters, metrics, and artifacts** to MLflow experiments
3. **Compare runs** using the MLflow UI
4. **Automate hyperparameter search** with Optuna, tracked by MLflow
5. **Register and version models** in the MLflow Model Registry
6. **Load a registered model** programmatically for inference

### Business Context

**Credit risk** is the risk that a borrower fails to repay a loan. Banks build credit-risk models to minimize expected losses, and machine learning classifiers can help predict whether a customer is likely to default.

In this lab you will work with the [German Credit Risk dataset](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data) (UCI), which contains anonymized profiles of 1 000 bank customers. Your job is to build a classifier that predicts whether a customer is **creditworthy** (`0`) or **at risk** (`1`).

### Why MLflow?

In production ML, tracking experiments is crucial. Without proper tracking:
- You can't reproduce results
- You lose track of which hyperparameters worked best
- Collaboration becomes chaotic
- Model deployment becomes error-prone

MLflow solves these problems by providing a unified platform for the ML lifecycle.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, start MLflow server, connect |
| 1 | Data Exploration | Load data, EDA, class balance analysis |
| 2 | Feature Engineering | Identify and encode categorical features |
| 3 | Baseline Model + MLflow Logging | Train a default Random Forest, log everything to MLflow |
| 4 | Manual Tuning + MLflow Comparison | Tweak hyperparameters by hand, compare runs in the UI |
| 5 | Automated Tuning with Optuna | Use Optuna for systematic search, nested MLflow runs |
| 6 | Model Registry | Register, version, and load the best model |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

Make sure that you have a virtual environment created for this lab (See README.md installation guide - Step 2). Select this venv for the notebook kernel.

In [ ]:
%pip install -r requirements.txt

### 0.2 Imports

In [ ]:
import pickle
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import mlflow
from mlflow import MlflowClient
import optuna

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42          # reproducibility
TEST_SIZE = 0.20           # 80/20 split
VAL_SIZE = 0.20            # 20 % of training set for validation

print(f"MLflow version: {mlflow.__version__}")
print(f"Optuna version: {optuna.__version__}")

### 0.3 Start the MLflow Tracking Server

Before running the rest of this notebook, **open a terminal**, activate this lab's virtual environment and start a local MLflow server:

```bash
mlflow server --host 127.0.0.1 --port 8080
```

Then open the UI in your browser at [http://127.0.0.1:8080](http://127.0.0.1:8080).

> 📖 **Docs:** [MLflow Tracking Server](https://mlflow.org/docs/latest/tracking/server.html)

💡 **Tip**: Keep this terminal open throughout the lab. The MLflow server must be running to track experiments.

In [ ]:
##############################  TODO  ##############################
# Set host and port to match the server you just started.
MLFLOW_HOST = "..."   # TODO
MLFLOW_PORT = "..."   # TODO
####################################################################

TRACKING_URI = f"http://{MLFLOW_HOST}:{MLFLOW_PORT}"
mlflow.set_tracking_uri(TRACKING_URI)
print(f"Tracking URI set to: {TRACKING_URI}")

---
## 1 · Data Exploration

### 1.1 Load the dataset

In [ ]:
df = pd.read_parquet("data/dataset.parquet")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

### 1.2 Identify the target

The goal is to predict whether a bank should grant credit to a customer.

**✏️ TODO:** Look at the column names and values. Which column is the target?

In [ ]:
##############################  TODO  ##############################
# Replace the empty string with the name of the target column.
target_field = "..."  # TODO
####################################################################

In [ ]:
# Rename and remap: 1 → 0 (creditworthy), 2 → 1 (at risk)
df = df.rename(columns={target_field: "risk"})
df["risk"] = df["risk"].map({1: 0, 2: 1})

# Separate features and target
y = df["risk"]
X = df.drop(columns=["risk"])

print(f"Features: {X.shape[1]}  |  Samples: {X.shape[0]}")
print(f"Target distribution:\n{y.value_counts(normalize=True).round(3)}")

This is a **binary classification** problem:
- `risk = 0` → customer is creditworthy
- `risk = 1` → customer is at risk of default

### 1.3 Exploratory Data Analysis

#### Correlation matrix (numeric features only)

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5,
            cbar_kws={"shrink": 0.7}, center=0)
plt.title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.show()

#### Class distribution

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
y.value_counts().plot(kind="bar", color=["g", "r"], ax=ax)
ax.set_xticklabels(["Creditworthy (0)", "At Risk (1)"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Target Class Distribution")
plt.tight_layout()
plt.show()

**✏️ Question 1 — Class imbalance**

a) Is this dataset balanced? Compute the exact class proportions.  
b) What risks does class imbalance pose for a classifier?  
c) Name **two** techniques you could use to handle imbalance (you don't need to implement them here).

---
*Your answer:*



---

**✏️ Question 2 — From binary to multi-level risk**

In practice, banks use more granular risk categories (e.g., AAA, AA, A, BBB, …). Given a binary classifier that outputs probabilities, how could you derive *n* risk levels from it? Give a concrete example with n = 4.

---
*Your answer:*



---

---
## 2 · Feature Engineering

We need to encode categorical features before training a tree-based model.

> 📖 **Docs:** [`sklearn.preprocessing.OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

In [6]:
# Separate numeric and categorical columns
numeric_feat = X.select_dtypes(include="number").columns.tolist()
categorical_feat = X.select_dtypes(exclude="number").columns.tolist()

print(f"Numeric features ({len(numeric_feat)}): {numeric_feat}")
print(f"Categorical features ({len(categorical_feat)}): {categorical_feat}")

**✏️ TODO:** Create a OneHotEncoder instance

In [8]:
##############################  TODO  ##############################
# Using sklearn's OneHotEncoder:
#   1. Create a OneHotEncoder instance (use handle_unknown="ignore"
#      so the encoder won't crash on unseen categories at inference).
#   2. fit_transform it on X[categorical_feat].
#   3. Build a DataFrame X_enc that combines the encoded columns
#      with the original numeric columns.
#
# Useful methods:
#   encoder.fit_transform(data)          → sparse/dense array
#   sparse_array.toarray()               → convert to dense numpy array
#   encoder.get_feature_names_out(input_features=...)   → column names

onehot_encoder = ...  # TODO: create the encoder
X_enc_array = ...     # TODO: fit_transform on categorical columns 

X_enc = pd.DataFrame( # TODO: build the final DataFrame.
    ... ,
    columns= ...,
)
X_enc[numeric_feat] = X[numeric_feat].values
####################################################################

print(f"Encoded feature matrix shape: {X_enc.shape}")
X_enc.head()

In [ ]:
# Save the encoder — we will log it as an MLflow artifact later
with open("data/one_hot_encoder.pkl", "wb") as f:
    pickle.dump(onehot_encoder, f)
print("Encoder saved to data/one_hot_encoder.pkl")

**✏️ Question 3 — One-hot encoding**

a) In one sentence, what does one-hot encoding do?  
b) How would `['Cat', 'Cat', 'Dog', 'Cat', 'Bird', 'Dog']` be transformed?  

---
*Your answer:*



---

---
## 3 · Baseline Model + MLflow Logging

### 3.1 Train / Test Split

> 📖 **Docs:** [`sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

**✏️ TODO:** Split `X_enc` and `y` into training and test sets. Use `stratify=y` so that the class proportions are preserved in both sets.

In [ ]:
##############################  TODO  ##############################
# Use train_test_split with test_size=TEST_SIZE, stratify=y,
# and random_state=RANDOM_STATE.
X_train, X_test, y_train, y_test = ...  # TODO
####################################################################

print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")
print(f"Train risk ratio: {y_train.mean():.3f}  |  Test risk ratio: {y_test.mean():.3f}")

### 3.2 Train a baseline Random Forest (default hyperparameters)

> 📖 **Docs:** [`sklearn.ensemble.RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

In [ ]:
rf_baseline = RandomForestClassifier(random_state=RANDOM_STATE)
rf_baseline.fit(X_train, y_train)

y_pred_proba = rf_baseline.predict_proba(X_test)[:, 1]
y_pred = rf_baseline.predict(X_test)

print(f"Baseline AUC:      {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Baseline F1-score: {f1_score(y_test, y_pred):.4f}")

### 3.3 Log the baseline run to MLflow

This is the core of this lab: we will log **parameters**, **metrics**, the **model**, and an **artifact** (our encoder) to MLflow.

> 📖 **Key MLflow functions:**
> - [`mlflow.set_experiment()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.set_experiment)
> - [`mlflow.start_run()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.start_run)
> - [`mlflow.log_param()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_param) / [`mlflow.log_params()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_params)
> - [`mlflow.log_metric()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_metric)
> - [`mlflow.sklearn.log_model()`](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html#mlflow.sklearn.log_model)
> - [`mlflow.log_artifact()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_artifact)

In [ ]:
EXPERIMENT_NAME = "credit-risk-classification"
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

with mlflow.start_run(run_name="baseline_random_forest"):
    # --- Log hyperparameters ---
    params = rf_baseline.get_params()
    mlflow.log_param("n_estimators", params["n_estimators"])
    mlflow.log_param("max_depth", params["max_depth"])
    mlflow.log_param("min_samples_leaf", params["min_samples_leaf"])
    mlflow.log_param("bootstrap", params["bootstrap"])

    # --- Log metrics ---
    mlflow.log_metric("auc", roc_auc_score(y_test, y_pred_proba))
    mlflow.log_metric("f1_score", f1_score(y_test, y_pred))

    # --- Log the model ---
    mlflow.sklearn.log_model(
        rf_baseline,
        name="model",
        registered_model_name="credit-risk-rf",
    )

    # --- Log the encoder as an artifact ---
    mlflow.log_artifact(local_path="data/one_hot_encoder.pkl")

    print("✅ Baseline run logged to MLflow.")

**👉 Now open the MLflow UI** ([http://127.0.0.1:8080](http://127.0.0.1:8080)) and explore your run.

**✏️ Question 4 — Navigate the MLflow UI**

Using the MLflow UI, answer the following:

a) What are the default values for `n_estimators`, `max_depth`, and `bootstrap` logged for the baseline run?  
b) Describe the path you followed in the UI to find them (e.g., *Experiments → … → …*).

---
*Your answer:*



---

---
## 4 · Manual Tuning + MLflow Comparison

Now let's see if we can improve the baseline by **manually** adjusting a few hyperparameters.

**✏️ TODO:** Choose your own values for the hyperparameters below. Run the cell **at least 2–3 times** with different configurations, then compare all runs in the MLflow UI.

> 💡 **Tip:** After running the cell multiple times, go to the MLflow UI, select multiple runs, and click **Compare** to see a side-by-side table of parameters and metrics.

In [ ]:
##############################  TODO  ##############################
# Try different values. Run this cell multiple times, changing
# the values each time to explore the hyperparameter space.
manual_params = {
    "n_estimators": 100,          # TODO: try 50, 100, 200, 500...
    "max_depth": None,            # TODO: try 3, 5, 10, None...
    "min_samples_leaf": 1,        # TODO: try 1, 5, 10...
    "bootstrap": True,            # TODO: try True, False
}
####################################################################

rf_manual = RandomForestClassifier(**manual_params, random_state=RANDOM_STATE)
rf_manual.fit(X_train, y_train)

y_pred_proba = rf_manual.predict_proba(X_test)[:, 1]
y_pred = rf_manual.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)

mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
with mlflow.start_run(run_name="manual_tuning"):
    mlflow.log_params(manual_params)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1_score", f1)
    mlflow.sklearn.log_model(
        rf_manual,
        name="model",
        registered_model_name="credit-risk-rf",
    )
    mlflow.log_artifact(local_path="data/one_hot_encoder.pkl")

print(f"AUC: {auc:.4f}  |  F1: {f1:.4f}")
print("✅ Manual tuning run logged.")

**✏️ Question 5 — Manual tuning observations**

a) Which hyperparameter combination gave you the best F1-score so far?  
b) Why is manual tuning tedious and potentially unreliable?

---
*Your answer:*



---

---
## 5 · Automated Tuning with Optuna

[Optuna](https://optuna.readthedocs.io/en/stable/) is an automatic hyperparameter optimization framework. It uses smart search strategies (e.g., Tree-structured Parzen Estimators) instead of brute-force grid search.

> 📖 **Docs:** [Optuna — Key Concepts](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/001_first.html)

### 5.1 Create a validation split

We split the **training set** further into train and validation. The validation set is used by Optuna to score each trial. The test set remains untouched until final evaluation.

In [ ]:
X_train_opt, X_val, y_train_opt, y_val = train_test_split(
    X_train, y_train,
    test_size=VAL_SIZE,
    stratify=y_train,
    random_state=RANDOM_STATE,
)
print(f"Optuna train: {X_train_opt.shape[0]}  |  Validation: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")

**✏️ Question 6 — Train / Validation / Test**

a) Explain in your own words the role of each split: train, validation, and test.  
b) What happens if the validation set and test set overlap?

---
*Your answer:*



---

### 5.2 Define the Optuna objective

**✏️ TODO:** Fill in the hyperparameter search ranges. Use the [Optuna `suggest_*` API](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html):
- `trial.suggest_int(name, low, high)`
- `trial.suggest_float(name, low, high)`
- `trial.suggest_categorical(name, choices)`

In [ ]:
def objective(trial):
    """Optuna objective: train a RF with suggested params, return F1."""

    ##############################  TODO  ##############################
    # Define the hyperparameter search space.
    params = {
        "n_estimators":    trial.suggest_int("n_estimators", low=..., high=...),
        "max_depth":       trial.suggest_int("max_depth", low=..., high=...),
        "max_features":    trial.suggest_categorical("max_features", [...]),
        "min_samples_leaf": trial.suggest_float("min_samples_leaf", low=..., high=...),
    }
    ####################################################################

    clf = RandomForestClassifier(**params, random_state=RANDOM_STATE)
    clf.fit(X_train_opt, y_train_opt)

    y_val_pred = clf.predict(X_val)
    y_val_proba = clf.predict_proba(X_val)[:, 1]

    score = f1_score(y_val, y_val_pred)

    # Log to MLflow (nested run — each trial becomes a child run)
    mlflow.log_params(params)
    mlflow.log_metric("val_f1_score", score)
    mlflow.log_metric("val_auc", roc_auc_score(y_val, y_val_proba))

    return score

### 5.3 Run the optimization

We run 30 trials within a parent MLflow run. Each trial is logged as a nested child run.

In [ ]:
study = optuna.create_study(direction="maximize")

mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

with mlflow.start_run(run_name="optuna_search"):
    study.optimize(objective, n_trials=30, timeout=600)

best_params = study.best_trial.params
print(f"\n🏆 Best trial F1: {study.best_trial.value:.4f}")
print(f"Best params: {best_params}")

### 5.4 Retrain with the best parameters on full training data

Now that Optuna has found the best hyperparameters, we retrain on the **combined** train + validation sets and evaluate on the **held-out test set**.

In [ ]:
# Recombine train + validation
X_train_full = pd.concat([X_train_opt, X_val])
y_train_full = pd.concat([y_train_opt, y_val])

# Train final model
rf_best = RandomForestClassifier(**best_params, random_state=RANDOM_STATE)
rf_best.fit(X_train_full, y_train_full)

# Evaluate on test set
y_test_pred = rf_best.predict(X_test)
y_test_proba = rf_best.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, y_test_proba)
final_f1 = f1_score(y_test, y_test_pred)

print(f"Final Test AUC:      {final_auc:.4f}")
print(f"Final Test F1-score: {final_f1:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Creditworthy", "At Risk"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Best Model")
plt.tight_layout()
plt.show()

In [ ]:
# Log the final best model to MLflow
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

with mlflow.start_run(run_name="best_model_final_eval"):
    mlflow.log_params(best_params)
    mlflow.log_metric("test_auc", final_auc)
    mlflow.log_metric("test_f1_score", final_f1)

    mlflow.sklearn.log_model(
        rf_best,
        artifact_path="model",
        registered_model_name="credit-risk-rf-tuned",
    )
    mlflow.log_artifact(local_path="data/one_hot_encoder.pkl")

    # Save the confusion matrix plot as an artifact
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Creditworthy", "At Risk"]).plot(ax=ax, cmap="Blues")
    fig.savefig("data/confusion_matrix.png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    mlflow.log_artifact("data/confusion_matrix.png")

    print("✅ Best model logged with confusion matrix artifact.")

---
## 6 · Model Registry

The MLflow **Model Registry** lets you manage model versions and assign lifecycle stages like *Staging* or *Production*.

> 📖 **Docs:** [MLflow Model Registry](https://mlflow.org/docs/latest/model-registry.html)

### 6.1 Load a model from Model Registry

**✏️ TODO:** Load the model using its model URI. The model URI is made from the model name and the model version (Hint: check the **Models** tab in the MLflow UI).
> 📖 **Docs:** [`mlflow.sklearn.load_model`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.sklearn.html#mlflow.sklearn.load_model)

In [ ]:
##############################  TODO  ##############################
# Load and test the model using the model URI
MODEL_NAME = ... # TODO
MODEL_VERSION = ... # TODO
loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")
####################################################################

preds = loaded_model.predict(X_test[:5])
print(f"Sample predictions: {preds}")

### 6.2 Manage model versions with aliases

Modern MLflow (>= 2.9) uses **model aliases** instead of the deprecated "Stages" system. An alias is a mutable, named reference to a specific model version — for example, you might tag your best model version as `champion`.

> 📖 **Docs:** [Model aliases](https://mlflow.org/docs/latest/model-registry.html#deploy-and-organize-models-with-aliases-and-tags)

In [ ]:
client = MlflowClient()

MODEL_VERSION_ALIAS = "champion"

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias=MODEL_VERSION_ALIAS,
    version=MODEL_VERSION,
)
print(f"✅ Alias {MODEL_VERSION_ALIAS} set on {MODEL_NAME} v{MODEL_VERSION}")

### 6.3 Load a model by alias

Now anyone can load the production model using the alias, without needing to know the specific version number or run ID:

In [ ]:
# Load the model using its alias
champion_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@{MODEL_VERSION_ALIAS}")

preds = champion_model.predict(X_test[:5])
print(f"Champion model predictions: {preds}")
print("\n🎉 You have successfully loaded a model from the MLflow Registry!")

**✏️ Question 7 — Why use a Model Registry?**

In a team of data scientists, why is a model registry useful compared to simply saving model files to a shared folder? Think about versioning, reproducibility, and deployment.

---
*Your answer:*



---

---
## Summary

In this lab, you learned to:

| Step | What you did | MLflow feature used |
|------|-------------|---------------------|
| Setup | Start a tracking server and connect to it | `mlflow server`, `set_tracking_uri()` |
| Baseline | Train a model and log params/metrics/model | `log_param()`, `log_metric()`, `sklearn.log_model()` |
| Manual tuning | Compare runs side by side | MLflow UI comparison view |
| Optuna | Automate search with nested runs | `start_run()` context nesting |
| Artifacts | Log files alongside runs | `log_artifact()` |
| Registry | Version and alias models | `MlflowClient.set_registered_model_alias()` |

**Next lab:** We will move from local MLflow to **Vertex AI on GCP** to track experiments and deploy models in the cloud.